# LESSON 6.2: Noise Models in Image Processing
## Image Restoration

In this lesson:
- Gaussian noise model and its parameters
- Rayleigh noise
- Erlang (Gamma) noise
- Exponential noise
- Uniform noise
- Salt-and-pepper (impulse) noise
- Periodic (sinusoidal) noise
- Estimating noise parameters from image data
- Effect of noise on image histograms and spectra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Introduction: Why Study Noise Models?

Noise in digital images arises during **image acquisition** and **transmission**:

### Sources of Noise:
- **Sensor noise**: Electronic fluctuations in CCD/CMOS sensors
- **Photon noise**: Random arrival of photons (especially at low light/dose)
- **Quantization noise**: Digitization of continuous intensity values
- **Transmission noise**: Errors during data transfer (e.g., wireless channels)

### Why model noise?
- Different noise types require **different filtering strategies**
- Mean filters work well for Gaussian noise but poorly for impulse noise
- Median filters excel at impulse noise but are less effective for Gaussian noise
- Knowing the noise model guides the choice of restoration method

### Gonzalez Chapter 5.2:
Noise is described by its **probability density function (PDF)** and characterized by its **mean** ($\mu$) and **variance** ($\sigma^2$).

In [ ]:
# Create a test image for noise experiments
def create_test_image(size=256):
    """
    Create a synthetic biomedical phantom image.
    
    Returns:
        2D numpy array of shape (size, size), values in [0, 255]
    """
    img = np.ones((size, size), dtype=np.float64) * 30
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    body = ((X - cx) / 100) ** 2 + ((Y - cy) / 80) ** 2 <= 1
    img[body] = 120
    
    organ1 = ((X - cx + 30) / 35) ** 2 + ((Y - cy + 10) / 45) ** 2 <= 1
    img[organ1] = 170
    
    organ2 = ((X - cx - 35) / 25) ** 2 + ((Y - cy - 15) / 30) ** 2 <= 1
    img[organ2] = 80
    
    for (sx, sy, sr) in [(cx-20, cy+30, 5), (cx+40, cy-25, 4), (cx-50, cy-20, 3)]:
        spot = (X - sx) ** 2 + (Y - sy) ** 2 <= sr ** 2
        img[spot] = 240
    
    return img


original = create_test_image(256)
plt.figure(figsize=(6, 6))
plt.imshow(original, cmap='gray', vmin=0, vmax=255)
plt.title('Original Image (Clean)', fontsize=13)
plt.colorbar()
plt.show()

---
## 2. Gaussian Noise

The most common noise model. The PDF is:

$$p(z) = \frac{1}{\sqrt{2\pi}\sigma} e^{-(z-\mu)^2 / (2\sigma^2)}$$

Where:
- $z$ = noise value (gray-level intensity)
- $\mu$ = **mean** of the noise
- $\sigma$ = **standard deviation** of the noise
- $\sigma^2$ = **variance**

### Properties:
- Approximately **70%** of values fall within $[\mu - \sigma, \mu + \sigma]$
- Approximately **95%** of values fall within $[\mu - 2\sigma, \mu + 2\sigma]$
- Approximately **99.7%** of values fall within $[\mu - 3\sigma, \mu + 3\sigma]$

### Sources:
- Electronic circuit noise in sensors
- Thermal noise in detectors
- Many physical processes (Central Limit Theorem)

In [ ]:
# Generate and visualize Gaussian noise
np.random.seed(42)

# Noise PDF
z = np.linspace(-100, 100, 500)
mu, sigma = 0, 20
pdf_gauss = (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-(z - mu)**2 / (2 * sigma**2))

# Generate noise and apply to image
noise_gauss = np.random.normal(mu, sigma, original.shape)
noisy_gauss = np.clip(original + noise_gauss, 0, 255)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# PDF
axes[0, 0].plot(z, pdf_gauss, 'b-', linewidth=2)
axes[0, 0].fill_between(z, pdf_gauss, alpha=0.3)
axes[0, 0].axvline(x=mu, color='r', linestyle='--', label=f'μ = {mu}')
axes[0, 0].axvline(x=mu+sigma, color='g', linestyle='--', alpha=0.7, label=f'μ±σ = ±{sigma}')
axes[0, 0].axvline(x=mu-sigma, color='g', linestyle='--', alpha=0.7)
axes[0, 0].set_title('Gaussian PDF', fontsize=12)
axes[0, 0].set_xlabel('z')
axes[0, 0].set_ylabel('p(z)')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Noise pattern
axes[0, 1].imshow(noise_gauss, cmap='gray')
axes[0, 1].set_title(f'Noise Pattern η(x,y)\nμ={mu}, σ={sigma}', fontsize=12)
axes[0, 1].axis('off')

# Noise histogram
axes[0, 2].hist(noise_gauss.ravel(), bins=100, density=True, alpha=0.7, color='steelblue')
axes[0, 2].plot(z, pdf_gauss, 'r-', linewidth=2, label='Theoretical PDF')
axes[0, 2].set_title('Noise Histogram vs PDF', fontsize=12)
axes[0, 2].set_xlabel('Noise value')
axes[0, 2].legend(fontsize=10)
axes[0, 2].grid(True, alpha=0.3)

# Original image
axes[1, 0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Original', fontsize=12)
axes[1, 0].axis('off')

# Noisy image
axes[1, 1].imshow(noisy_gauss, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('Gaussian Noisy Image', fontsize=12)
axes[1, 1].axis('off')

# Difference
axes[1, 2].imshow(np.abs(original - noisy_gauss), cmap='hot', vmin=0, vmax=80)
axes[1, 2].set_title('|Original - Noisy|', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Gaussian Noise Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Noise statistics: mean = {noise_gauss.mean():.2f}, std = {noise_gauss.std():.2f}")
print(f"Theoretical: mean = {mu}, std = {sigma}")

In [ ]:
# Effect of different sigma values
sigma_values = [5, 15, 30, 60]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, sigma in enumerate(sigma_values):
    np.random.seed(42)
    noise = np.random.normal(0, sigma, original.shape)
    noisy = np.clip(original + noise, 0, 255)
    
    axes[0, i].imshow(noisy, cmap='gray', vmin=0, vmax=255)
    psnr = 10 * np.log10(255**2 / np.mean((original - noisy)**2))
    axes[0, i].set_title(f'σ = {sigma}\nPSNR = {psnr:.1f} dB', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].hist(noise.ravel(), bins=80, density=True, alpha=0.7, color='steelblue')
    z_range = np.linspace(-3*sigma, 3*sigma, 200)
    pdf = (1/(np.sqrt(2*np.pi)*sigma)) * np.exp(-z_range**2 / (2*sigma**2))
    axes[1, i].plot(z_range, pdf, 'r-', linewidth=2)
    axes[1, i].set_title(f'Noise Distribution (σ={sigma})', fontsize=11)
    axes[1, i].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Noisy Image', fontsize=12)
axes[1, 0].set_ylabel('Noise Histogram', fontsize=12)

plt.suptitle('Effect of Gaussian Noise Level (σ) on Image Quality',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Rayleigh Noise

$$p(z) = \begin{cases} \frac{2}{b}(z - a)\, e^{-(z-a)^2/b} & \text{for } z \geq a \\ 0 & \text{for } z < a \end{cases}$$

Where:
- $a$ = location parameter (minimum value)
- $b$ = scale parameter

### Statistics:
- Mean: $\mu = a + \sqrt{\pi b / 4}$
- Variance: $\sigma^2 = \frac{b(4 - \pi)}{4}$

### Properties:
- **Skewed** distribution (asymmetric, right-tailed)
- Useful for modeling noise in **range imaging** and **MRI magnitude images**
- The Rician distribution (used in MRI) is related to Rayleigh

In [ ]:
# Generate and visualize Rayleigh noise
np.random.seed(42)

a = 0
b = 400  # scale parameter

# Generate Rayleigh noise
noise_rayleigh = a + np.random.rayleigh(scale=np.sqrt(b/2), size=original.shape)

# Shift to zero-mean for additive noise
noise_rayleigh_centered = noise_rayleigh - noise_rayleigh.mean()
noisy_rayleigh = np.clip(original + noise_rayleigh_centered, 0, 255)

# Theoretical PDF
z = np.linspace(0, 60, 500)
pdf_rayleigh = (2/b) * (z - a) * np.exp(-(z - a)**2 / b)
pdf_rayleigh[z < a] = 0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(z, pdf_rayleigh, 'b-', linewidth=2)
axes[0].fill_between(z, pdf_rayleigh, alpha=0.3)
axes[0].set_title(f'Rayleigh PDF (a={a}, b={b})', fontsize=12)
axes[0].set_xlabel('z')
axes[0].set_ylabel('p(z)')
axes[0].grid(True, alpha=0.3)

axes[1].imshow(noisy_rayleigh, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Rayleigh Noisy Image', fontsize=12)
axes[1].axis('off')

axes[2].hist(noise_rayleigh.ravel(), bins=80, density=True, alpha=0.7, color='steelblue')
axes[2].plot(z, pdf_rayleigh, 'r-', linewidth=2, label='Theoretical PDF')
axes[2].set_title('Rayleigh Noise Distribution', fontsize=12)
axes[2].set_xlabel('Noise value')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Rayleigh Noise Model (Skewed Distribution)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Key property: Rayleigh noise is SKEWED (asymmetric)")
print(f"Mean: {noise_rayleigh.mean():.2f}, Std: {noise_rayleigh.std():.2f}")

---
## 4. Erlang (Gamma) Noise

$$p(z) = \begin{cases} \frac{a^b z^{b-1}}{(b-1)!} e^{-az} & \text{for } z \geq 0 \\ 0 & \text{for } z < 0 \end{cases}$$

Where:
- $a > 0$ = rate parameter
- $b$ = positive integer (shape parameter)

### Statistics:
- Mean: $\mu = b / a$
- Variance: $\sigma^2 = b / a^2$

### Properties:
- When $b = 1$, this reduces to the **Exponential** distribution
- Skewed distribution, becomes more symmetric as $b$ increases
- Used in **laser imaging** and other photon-counting applications

In [ ]:
# Generate and visualize Erlang (Gamma) noise for different parameters
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Different shape parameters
params = [(1, 0.1, 'b=1 (Exponential)'), 
          (3, 0.1, 'b=3'),
          (10, 0.1, 'b=10')]

z = np.linspace(0.01, 150, 500)

for i, (b, a, label) in enumerate(params):
    # Generate gamma noise
    noise = np.random.gamma(shape=b, scale=1/a, size=original.shape)
    noise_centered = noise - noise.mean()
    noisy = np.clip(original + noise_centered, 0, 255)
    
    # Theoretical PDF using scipy-free formula
    from math import factorial
    if b <= 20:  # avoid overflow
        pdf = (a**b * z**(b-1) / factorial(b-1)) * np.exp(-a * z)
    else:
        pdf = np.zeros_like(z)
    
    axes[0, i].plot(z, pdf, 'b-', linewidth=2)
    axes[0, i].fill_between(z, pdf, alpha=0.3)
    axes[0, i].set_title(f'Erlang PDF ({label})', fontsize=12)
    axes[0, i].set_xlabel('z')
    axes[0, i].set_ylabel('p(z)')
    axes[0, i].grid(True, alpha=0.3)
    
    axes[1, i].imshow(noisy, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Noisy Image ({label})', fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Erlang (Gamma) Noise: Effect of Shape Parameter b',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("b=1: Exponential distribution (highly skewed)")
print("b=3: Moderately skewed")
print("b=10: More symmetric, approaches Gaussian shape")

---
## 5. Exponential Noise

A special case of Erlang with $b = 1$:

$$p(z) = \begin{cases} a \, e^{-az} & \text{for } z \geq 0 \\ 0 & \text{for } z < 0 \end{cases}$$

### Statistics:
- Mean: $\mu = 1/a$
- Variance: $\sigma^2 = 1/a^2$

### Properties:
- Memoryless property: $P(Z > s+t | Z > s) = P(Z > t)$
- Models the **time between events** in a Poisson process
- Common in **laser-based imaging** systems

In [ ]:
# Generate and visualize Exponential noise
np.random.seed(42)

a = 0.05  # rate parameter
noise_exp = np.random.exponential(scale=1/a, size=original.shape)
noise_exp_centered = noise_exp - noise_exp.mean()
noisy_exp = np.clip(original + noise_exp_centered, 0, 255)

z = np.linspace(0.01, 100, 500)
pdf_exp = a * np.exp(-a * z)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(z, pdf_exp, 'b-', linewidth=2)
axes[0].fill_between(z, pdf_exp, alpha=0.3)
axes[0].set_title(f'Exponential PDF (a={a})', fontsize=12)
axes[0].set_xlabel('z')
axes[0].set_ylabel('p(z)')
axes[0].grid(True, alpha=0.3)

axes[1].imshow(noisy_exp, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Exponential Noisy Image', fontsize=12)
axes[1].axis('off')

axes[2].hist(noise_exp.ravel(), bins=80, density=True, alpha=0.7, color='steelblue')
axes[2].plot(z, pdf_exp, 'r-', linewidth=2, label='Theoretical PDF')
axes[2].set_title('Noise Histogram vs PDF', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Exponential Noise Model',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Uniform Noise

$$p(z) = \begin{cases} \frac{1}{b - a} & \text{if } a \leq z \leq b \\ 0 & \text{otherwise} \end{cases}$$

### Statistics:
- Mean: $\mu = \frac{a + b}{2}$
- Variance: $\sigma^2 = \frac{(b - a)^2}{12}$

### Properties:
- All values in $[a, b]$ are equally likely
- Used as a **baseline** or reference noise model
- Models **quantization noise** (the error introduced by digitizing continuous values)

In [ ]:
# Generate and visualize Uniform noise
np.random.seed(42)

a_unif, b_unif = -30, 30  # range of noise
noise_uniform = np.random.uniform(a_unif, b_unif, original.shape)
noisy_uniform = np.clip(original + noise_uniform, 0, 255)

z = np.linspace(-50, 50, 500)
pdf_uniform = np.where((z >= a_unif) & (z <= b_unif), 1.0/(b_unif - a_unif), 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(z, pdf_uniform, 'b-', linewidth=2)
axes[0].fill_between(z, pdf_uniform, alpha=0.3)
axes[0].set_title(f'Uniform PDF (a={a_unif}, b={b_unif})', fontsize=12)
axes[0].set_xlabel('z')
axes[0].set_ylabel('p(z)')
axes[0].set_ylim([0, 0.025])
axes[0].grid(True, alpha=0.3)

axes[1].imshow(noisy_uniform, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Uniform Noisy Image', fontsize=12)
axes[1].axis('off')

axes[2].hist(noise_uniform.ravel(), bins=80, density=True, alpha=0.7, color='steelblue')
axes[2].plot(z, pdf_uniform, 'r-', linewidth=2, label='Theoretical PDF')
axes[2].set_title('Noise Histogram vs PDF', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Uniform Noise Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Salt-and-Pepper (Impulse) Noise

$$p(z) = \begin{cases} P_s & \text{for } z = 255 \text{ (salt: white)} \\ P_p & \text{for } z = 0 \text{ (pepper: black)} \\ 1 - P_s - P_p & \text{for } z = \text{original value} \end{cases}$$

Where:
- $P_s$ = probability of a salt (white) pixel
- $P_p$ = probability of a pepper (black) pixel
- Total corruption probability: $P_s + P_p$

### Properties:
- **Impulse noise**: affects individual pixels randomly
- Corrupted pixels take **extreme values** (0 or 255)
- Caused by **faulty sensors**, **transmission errors**, or **dead pixels**
- **Mean filters are ineffective** — median filter is the standard remedy

### Clinical Relevance:
- Dead pixels in CCD detectors for X-ray or mammography
- Data dropouts in telemetry from remote imaging systems

In [ ]:
# Generate and visualize Salt-and-Pepper noise

def add_salt_pepper_noise(image, prob_salt=0.02, prob_pepper=0.02):
    """
    Add salt-and-pepper (impulse) noise to an image.
    
    Parameters:
        image: 2D numpy array
        prob_salt: probability of a white (salt) pixel
        prob_pepper: probability of a black (pepper) pixel
    Returns:
        Noisy image
    """
    noisy = image.copy()
    total = image.size
    
    # Salt (white pixels)
    num_salt = int(prob_salt * total)
    coords_salt = [np.random.randint(0, i, num_salt) for i in image.shape]
    noisy[coords_salt[0], coords_salt[1]] = 255
    
    # Pepper (black pixels)
    num_pepper = int(prob_pepper * total)
    coords_pepper = [np.random.randint(0, i, num_pepper) for i in image.shape]
    noisy[coords_pepper[0], coords_pepper[1]] = 0
    
    return noisy


np.random.seed(42)

# Different corruption levels
probs = [(0.01, 0.01), (0.05, 0.05), (0.1, 0.1), (0.2, 0.2)]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, (ps, pp) in enumerate(probs):
    noisy_sp = add_salt_pepper_noise(original, ps, pp)
    
    axes[0, i].imshow(noisy_sp, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(f'P_s={ps}, P_p={pp}\nTotal: {(ps+pp)*100:.0f}%', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].hist(noisy_sp.ravel(), bins=256, range=(0, 255), density=True,
                    alpha=0.7, color='steelblue')
    axes[1, i].set_title('Histogram', fontsize=11)
    axes[1, i].set_xlabel('Intensity')
    axes[1, i].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Noisy Image', fontsize=12)
axes[1, 0].set_ylabel('Histogram', fontsize=12)

plt.suptitle('Salt-and-Pepper (Impulse) Noise at Different Corruption Levels',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice the spikes at 0 (pepper) and 255 (salt) in the histograms.")
print("Salt-and-pepper noise does NOT affect all pixels — only a random subset.")

---
## 8. Periodic (Sinusoidal) Noise

$$\eta(x,y) = A \sin(2\pi u_0 x / M + 2\pi v_0 y / N + \phi)$$

Where:
- $A$ = amplitude of the periodic interference
- $(u_0, v_0)$ = spatial frequencies of the interference pattern
- $\phi$ = phase offset
- $M \times N$ = image dimensions

### Properties:
- Produces a **repeating pattern** overlaid on the image
- In the frequency domain, appears as **bright impulse pairs** at $(\pm u_0, \pm v_0)$
- Can be removed using **notch filters** or **band-reject filters** in the frequency domain

### Sources:
- Electrical or mechanical interference during image acquisition
- Aliasing artifacts in CT or MRI
- Moiré patterns from scanning printed images

In [ ]:
# Generate and visualize periodic noise
M, N = original.shape
Y, X = np.mgrid[0:M, 0:N]

# Single frequency periodic noise
u0, v0 = 30, 40  # spatial frequencies
A = 30  # amplitude
noise_periodic = A * np.sin(2 * np.pi * u0 * X / N + 2 * np.pi * v0 * Y / M)
noisy_periodic = np.clip(original + noise_periodic, 0, 255)

# Spectrum of noisy image
F_noisy = np.fft.fftshift(np.fft.fft2(noisy_periodic))
F_original = np.fft.fftshift(np.fft.fft2(original))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(noise_periodic, cmap='gray')
axes[0, 1].set_title(f'Periodic Noise\n$u_0$={u0}, $v_0$={v0}, A={A}', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(noisy_periodic, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title('Corrupted Image', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(F_original)), cmap='hot')
axes[1, 0].set_title('Spectrum: Original', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(noise_periodic)))), cmap='hot')
axes[1, 1].set_title('Spectrum: Noise Only\n(impulse pairs visible)', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(np.log1p(np.abs(F_noisy)), cmap='hot')
axes[1, 2].set_title('Spectrum: Corrupted\n(noise spikes + image spectrum)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Periodic Noise: Spatial and Frequency Domain View',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The periodic noise appears as BRIGHT DOTS in the frequency spectrum.")
print("These can be surgically removed using NOTCH FILTERS.")

In [ ]:
# Multiple periodic noise components
noise_multi = (20 * np.sin(2 * np.pi * 20 * X / N + 2 * np.pi * 25 * Y / M) +
               15 * np.sin(2 * np.pi * 50 * X / N) +
               10 * np.cos(2 * np.pi * 40 * Y / M))
noisy_multi = np.clip(original + noise_multi, 0, 255)

F_multi = np.fft.fftshift(np.fft.fft2(noisy_multi))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(noise_multi, cmap='gray')
axes[0].set_title('Multiple Periodic Noise Components', fontsize=12)
axes[0].axis('off')

axes[1].imshow(noisy_multi, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Corrupted Image', fontsize=12)
axes[1].axis('off')

axes[2].imshow(np.log1p(np.abs(F_multi)), cmap='hot')
axes[2].set_title('Spectrum: Multiple Noise Frequencies', fontsize=12)
axes[2].axis('off')

plt.suptitle('Multiple Periodic Interference Patterns',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each sinusoidal component creates a PAIR of impulses in the spectrum.")
print("With multiple components, we see multiple pairs that need separate notch filters.")

---
## 9. Comparison of All Noise Models

| Noise Type | PDF Shape | Mean | Variance | Best Filter |
|---|---|---|---|---|
| **Gaussian** | Symmetric bell | $\mu$ | $\sigma^2$ | Mean/Gaussian filter |
| **Rayleigh** | Right-skewed | $a + \sqrt{\pi b/4}$ | $b(4-\pi)/4$ | Adaptive filter |
| **Erlang** | Right-skewed | $b/a$ | $b/a^2$ | Adaptive filter |
| **Exponential** | Monotone decreasing | $1/a$ | $1/a^2$ | Geometric mean |
| **Uniform** | Flat | $(a+b)/2$ | $(b-a)^2/12$ | Mean filter |
| **Salt & Pepper** | Impulses at 0, 255 | — | — | **Median filter** |
| **Periodic** | Sinusoidal | 0 | $A^2/2$ | **Notch filter** |

In [ ]:
# Side-by-side comparison of all noise types
np.random.seed(42)

# Generate all noise types
gauss = np.clip(original + np.random.normal(0, 20, original.shape), 0, 255)

rayleigh_n = np.random.rayleigh(scale=15, size=original.shape)
rayleigh_img = np.clip(original + rayleigh_n - rayleigh_n.mean(), 0, 255)

gamma_n = np.random.gamma(shape=3, scale=7, size=original.shape)
gamma_img = np.clip(original + gamma_n - gamma_n.mean(), 0, 255)

exp_n = np.random.exponential(scale=20, size=original.shape)
exp_img = np.clip(original + exp_n - exp_n.mean(), 0, 255)

unif = np.clip(original + np.random.uniform(-30, 30, original.shape), 0, 255)

sp = add_salt_pepper_noise(original, 0.05, 0.05)

Y_grid, X_grid = np.mgrid[0:M, 0:N]
periodic = np.clip(original + 30*np.sin(2*np.pi*30*X_grid/N + 2*np.pi*30*Y_grid/M), 0, 255)

noise_images = [original, gauss, rayleigh_img, gamma_img, exp_img, unif, sp, periodic]
noise_labels = ['Original', 'Gaussian\n(σ=20)', 'Rayleigh', 'Erlang\n(b=3)',
                'Exponential', 'Uniform\n(±30)', 'Salt&Pepper\n(10%)', 'Periodic']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, (img, label) in enumerate(zip(noise_images, noise_labels)):
    row, col = i // 4, i % 4
    axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[row, col].set_title(label, fontsize=12)
    axes[row, col].axis('off')

plt.suptitle('Comparison of All Noise Models Applied to the Same Image',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Estimating Noise Parameters from Image Data

In practice, the noise parameters are **unknown** and must be estimated from the image.

### Method: Select a flat (constant intensity) region

If we can identify a **homogeneous region** $S$ in the image where the true intensity should be constant:

$$\hat{\mu} = \frac{1}{|S|} \sum_{(x,y) \in S} g(x,y)$$

$$\hat{\sigma}^2 = \frac{1}{|S|} \sum_{(x,y) \in S} (g(x,y) - \hat{\mu})^2$$

The histogram of the region approximates the noise PDF.

In [ ]:
# Estimate noise parameters from a flat region
np.random.seed(42)

# Create noisy image with known noise
true_sigma = 20
noisy = np.clip(original + np.random.normal(0, true_sigma, original.shape), 0, 255)

# Select a flat region (we know the background is approximately constant)
# Region in the dark background area
region = noisy[10:60, 10:60]  # 50x50 patch in the background

# Estimate noise parameters
estimated_mean = np.mean(region)
estimated_std = np.std(region)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Show the noisy image with the selected region highlighted
axes[0].imshow(noisy, cmap='gray', vmin=0, vmax=255)
rect = plt.Rectangle((10, 10), 50, 50, linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title('Noisy Image\n(red box = estimation region)', fontsize=12)
axes[0].axis('off')

# Show the region
axes[1].imshow(region, cmap='gray')
axes[1].set_title(f'Flat Region (50×50)\nMean={estimated_mean:.1f}, Std={estimated_std:.1f}', fontsize=12)
axes[1].axis('off')

# Histogram of the region
axes[2].hist(region.ravel(), bins=40, density=True, alpha=0.7, color='steelblue',
             label='Region histogram')
z = np.linspace(region.min(), region.max(), 200)
pdf_fit = (1/(np.sqrt(2*np.pi)*estimated_std)) * np.exp(-(z - estimated_mean)**2 / (2*estimated_std**2))
axes[2].plot(z, pdf_fit, 'r-', linewidth=2, label=f'Gaussian fit\nσ̂={estimated_std:.1f}')
axes[2].set_title('Noise Parameter Estimation', fontsize=12)
axes[2].set_xlabel('Intensity')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Estimating Noise Parameters from a Homogeneous Region',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"True noise parameters: μ = 0, σ = {true_sigma}")
print(f"Estimated from region: μ̂ = {estimated_mean:.2f}, σ̂ = {estimated_std:.2f}")
print(f"Background intensity ≈ 30, so estimated μ̂ ≈ 30 + 0 = 30")

In [ ]:
# Effect of noise on image histograms
np.random.seed(42)

noisy_low = np.clip(original + np.random.normal(0, 10, original.shape), 0, 255)
noisy_high = np.clip(original + np.random.normal(0, 40, original.shape), 0, 255)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, (img, label) in enumerate([(original, 'Original'), 
                                   (noisy_low, 'Low Noise (σ=10)'),
                                   (noisy_high, 'High Noise (σ=40)')]):
    axes[0, i].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(label, fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].hist(img.ravel(), bins=128, range=(0, 255), density=True, 
                    alpha=0.7, color='steelblue')
    axes[1, i].set_title(f'Histogram', fontsize=11)
    axes[1, i].set_xlabel('Intensity')
    axes[1, i].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Histogram', fontsize=12)

plt.suptitle('Effect of Noise on Image Histogram\n'
             'Noise broadens histogram peaks and reduces contrast',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observation: Gaussian noise BROADENS histogram peaks.")
print("As σ increases, distinct peaks merge and the histogram flattens.")
print("This reduces the ability to distinguish different tissue types.")

---
## Summary

What we learned:

1. **Gaussian noise** has a symmetric bell-shaped PDF: $p(z) = \frac{1}{\sqrt{2\pi}\sigma} e^{-(z-\mu)^2/(2\sigma^2)}$. Most common in sensor electronics.

2. **Rayleigh noise** is right-skewed, used in range imaging and related to MRI Rician noise.

3. **Erlang (Gamma) noise** with shape parameter $b$ ranges from exponential ($b=1$) to near-Gaussian (large $b$).

4. **Exponential noise** is a special case of Erlang ($b=1$), common in laser imaging.

5. **Uniform noise** is equally likely across a range $[a,b]$, models quantization error.

6. **Salt-and-pepper noise** randomly sets pixels to 0 (pepper) or 255 (salt). Best removed by **median filtering**.

7. **Periodic noise** creates sinusoidal patterns visible as impulse pairs in the frequency spectrum. Removed by **notch filters**.

8. **Noise parameters** can be estimated from homogeneous (flat) image regions by computing the mean and variance of the region.

9. The **noise model** determines the optimal restoration strategy — there is no single filter that works best for all noise types.